<a href="https://colab.research.google.com/github/DanyRE4020/Concentracion_IA_avanzada/blob/main/Notebook6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [3]:
archivo = "/content/meteorología_2025.csv"
data = pd.read_csv(archivo, skiprows=9)

In [4]:
data.head()

,date,id_station,id_parameter,valor,unit
0,2025-01-01 00:00:00,ACO,TMP,8.7,5
1,2025-01-01 00:00:00,ACO,RH,28.0,6
2,2025-01-01 00:00:00,ACO,WSP,2.4,3
3,2025-01-01 00:00:00,ACO,WDR,2.0,4
4,2025-01-01 00:00:00,AJM,TMP,9.6,5


In [5]:
data.shape

(1191360, 5)

In [6]:
data.columns

Index(['date', 'id_station', 'id_parameter', 'valor', 'unit'], dtype='object')

In [7]:
data["date"] = pd.to_datetime(data["date"])
data["valor"] = pd.to_numeric(data["valor"], errors="coerce")

In [8]:
estacion = data[(data["id_station"] == "UAX") & (data["id_parameter"].isin(["RH", "WSP"]))]
estacion.head()

,date,id_station,id_parameter,valor,unit
121,2025-01-01 00:00:00,UAX,RH,41.0,6
122,2025-01-01 00:00:00,UAX,WSP,1.1,3
257,2025-01-01 01:00:00,UAX,RH,45.0,6
258,2025-01-01 01:00:00,UAX,WSP,1.1,3
393,2025-01-01 02:00:00,UAX,RH,46.0,6


In [9]:
estacion = estacion.pivot(index="date", columns="id_parameter", values="valor").reset_index()
estacion = estacion.dropna(subset=["RH", "WSP"])
estacion.head()

id_parameter,date,RH,WSP
0,2025-01-01 00:00:00,41.0,1.1
1,2025-01-01 01:00:00,45.0,1.1
2,2025-01-01 02:00:00,46.0,1.0
3,2025-01-01 03:00:00,55.0,0.2
4,2025-01-01 04:00:00,55.0,1.1


In [10]:
len(estacion)

8757

In [11]:
# Definir eventos
estacion["humedad_alta"] = estacion["RH"] >= 70
estacion["viento_tranquilo"] = estacion["WSP"] <= 1
estacion.head()

id_parameter,date,RH,WSP,humedad_alta,viento_tranquilo
0,2025-01-01 00:00:00,41.0,1.1,False,False
1,2025-01-01 01:00:00,45.0,1.1,False,False
2,2025-01-01 02:00:00,46.0,1.0,False,True
3,2025-01-01 03:00:00,55.0,0.2,False,True
4,2025-01-01 04:00:00,55.0,1.1,False,False


In [12]:
# Probabilidades
prob_humedad_alta = estacion["humedad_alta"].mean()
prob_humedad_alta

np.float64(0.35834189791024323)

In [13]:
prob_viento_tranquilo = estacion["viento_tranquilo"].mean()
prob_viento_tranquilo

np.float64(0.22130866735183283)

In [14]:
ambos_eventos = estacion["humedad_alta"] & estacion["viento_tranquilo"]
prob_ambos = ambos_eventos.mean()
prob_ambos

np.float64(0.14913783259107)

In [15]:
horas_humedad_alta = estacion[estacion["humedad_alta"]]
prob_viento_dado_humedad = horas_humedad_alta["viento_tranquilo"].mean()
prob_viento_dado_humedad

np.float64(0.41618865519439135)

# Tabla final de resultados

In [16]:
resultados = pd.DataFrame({
    "Evento": [
        "Humedad alta",
        "Viento tranquilo",
        "Humedad alta y viento tranquilo",
        "Viento tranquilo dado que existe humedad alta"
    ],
    "Probabilidad": [
        prob_humedad_alta,
        prob_viento_tranquilo,
        prob_ambos,
        prob_viento_dado_humedad
    ]
})

resultados["Porcentaje"] = (resultados["Probabilidad"] * 100).round(2)
resultados

,Evento,Probabilidad,Porcentaje
0,Humedad alta,0.358342,35.83
1,Viento tranquilo,0.221309,22.13
2,Humedad alta y viento tranquilo,0.149138,14.91
3,Viento tranquilo dado que existe humedad alta,0.416189,41.62


## Interpretación

En la estación UAX, la probabilidad de humedad alta es aproximadamente 35.83% y la probabilidad de viento tranquilo es 22.13%. La probabilidad de que ambos eventos sucedan en la misma hora es 14.91%. Cuando ya se sabe que existe humedad alta, la probabilidad de que el viento sea tranquilo aumenta a 41.62%.